# Earth Engine access check

**Do this before Day 1 of the course.** It takes about two minutes and it is the only way to be
sure your Earth Engine account actually works.

### What to do

1. Open the [Google Cloud Earth Engine page](https://console.cloud.google.com/earth-engine)
   and copy your **project ID** (the ID, not the display name — they are usually different).
2. Paste it into the cell below where it says `PROJECT_ID = 'ee-yourname'`.
3. Run the cell. When prompted, sign in with the **same Google account** the project belongs to.

### What you are looking for

The last line should read **ALL CHECKS PASSED**.

If it does not, the notebook will tell you which step failed and what to do about it. If you
cannot resolve it, email your instructor the **whole** error message — a screenshot of all the
red text, not just the last line — along with your project ID. Please do this before Day 1
rather than on the morning of it.

In [ ]:
# ============================================================
#  Earth Engine access check
#  Put your project ID on the next line, then run this cell.
# ============================================================

PROJECT_ID = 'ee-yourname'      # <-- CHANGE THIS

# ------------------------------------------------------------
import sys, subprocess

def ok(msg):   print(f'  [ OK ]  {msg}')
def bad(msg):  print(f'  [FAIL]  {msg}')

print('Earth Engine access check')
print('=' * 52)

# 1. library present
try:
    import ee
    ok(f'earthengine-api is installed (version {ee.__version__})')
except ImportError:
    print('  installing earthengine-api ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'earthengine-api'])
    import ee
    ok('earthengine-api installed')

# 2. did you change the placeholder?
if PROJECT_ID == 'ee-yourname':
    bad('You have not set PROJECT_ID yet.')
    print('\n        Open https://console.cloud.google.com/earth-engine')
    print('        copy your project ID, paste it above, and run again.')
    raise SystemExit

# 3. authenticate
try:
    ee.Authenticate()
    ok('signed in to Google')
except Exception as e:
    bad(f'authentication failed: {e}')
    raise SystemExit

# 4. initialise with the project
try:
    ee.Initialize(project=PROJECT_ID)
    ok(f'connected to project "{PROJECT_ID}"')
except Exception as e:
    msg = str(e)
    bad('could not connect to your project')
    print(f'\n        {msg[:200]}\n')
    if 'not found' in msg or 'deleted' in msg:
        print('        -> The project ID is wrong. Copy the ID (not the display')
        print('           name) from console.cloud.google.com/earth-engine')
    elif 'permission' in msg.lower():
        print('        -> This looks like a university account with the API blocked.')
        print('           Register again using a personal @gmail.com address.')
    elif 'has not been used' in msg or 'disabled' in msg:
        print('        -> The Earth Engine API is not enabled. Open the link in the')
        print('           message above and click Enable, then wait a minute.')
    else:
        print('        -> Send this whole message to your instructor before Day 1.')
    raise SystemExit

# 5. can we actually read data?
try:
    n = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterDate('2024-01-01', '2024-03-01')
         .filterBounds(ee.Geometry.Point([100.5, 18.8]))
         .size().getInfo())
    ok(f'read the Sentinel-2 catalog ({n} scenes over Nan, Jan-Feb 2024)')
except Exception as e:
    bad(f'could not read data: {str(e)[:150]}')
    raise SystemExit

# 6. can we run a computation?
try:
    aoi = (ee.FeatureCollection('FAO/GAUL/2015/level1')
           .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
           .filter(ee.Filter.eq('ADM1_NAME', 'Nan')))
    ha = aoi.geometry().area(maxError=100).divide(1e4).getInfo()
    ok(f'ran a computation (Nan province = {ha:,.0f} ha)')
except Exception as e:
    bad(f'computation failed: {str(e)[:150]}')
    raise SystemExit

# 7. geemap for the interactive maps
try:
    import geemap
    ok(f'geemap is available (version {geemap.__version__})')
except ImportError:
    print('  installing geemap ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'geemap'])
    ok('geemap installed')

print('=' * 52)
print('  ALL CHECKS PASSED - you are ready for Day 1.')
print()
print(f'  Your project ID is: {PROJECT_ID}')
print('  Write it down. You will type it in every session.')

### Optional — check that interactive maps display

Run this too if you like. You should see a map with the outline of Nan province on it. If the map area stays blank, that is usually a browser extension blocking it rather than a problem with your account.

In [ ]:
# Optional: prove the maps work too.
import geemap

aoi = (ee.FeatureCollection('FAO/GAUL/2015/level1')
       .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
       .filter(ee.Filter.eq('ADM1_NAME', 'Nan')))

Map = geemap.Map()
Map.centerObject(aoi, 8)
Map.addLayer(aoi.style(color='D97757', fillColor='D9775722', width=2), {}, 'Nan province')
Map

---

### Done?

Write your project ID somewhere you can copy it from — a note on your phone works. You will type
it at the start of every session for the rest of the course.

Then have a think about **which province you want to study**. Every lab lets you swap in your own
area, and the course is much more interesting when the map is somewhere you actually care about.